# 7.14 少量錯誤答案會被學走嗎？


1+1真實答案2，資料卻標3，交叉熵會自行發現標錯嗎？它只取我們提供的目標機率；模型越相信錯標，這份資料的代價越低。

用四候選「0、1、2、3」，ID恰好0到3，這是獨立分類算例，與byte字表的數字字符ID不同。先給真候選2分數3，錯候選3分數1，再只提高錯候選到5。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from torch.nn import functional as F

logits = torch.tensor([[0.0, 0.0, 3.0, 1.0]])
for label in [2, 3]:
    loss = F.cross_entropy(logits, torch.tensor([label]))
    print("提供的答案ID", label, "代價", loss.item())
wrong_favored = torch.tensor([[0.0, 0.0, 3.0, 5.0]])
print("提高錯字分數後，錯標代價", F.cross_entropy(wrong_favored, torch.tensor([3])).item())

提供的答案ID 2 代價 0.21099762618541718
提供的答案ID 3 代價 2.2109975814819336
提高錯字分數後，錯標代價 0.13872769474983215


輸入四分數，真實候選2得3、錯候選3得1。softmax後2約0.810，3約0.110，所以提供label2時代價約0.211、label3時約2.211。最後把錯候選分數提高到5，錯誤標籤的代價降到約0.139，儘管真實答案2的相對支持變少。交叉熵完整遵從輸入label，沒有自動檢查1+1的數學事實。

真實訓練中，錯誤答案也會產生清楚梯度，推動這類前文更常接錯字。如果錯例反覆出現或答案很長，其有效token影響可能超出筆數比例，見 [5.16](https://birdhackor.github.io/tiny-perceptron-vlm/5.16.html)。少量錯誤有沒有可觀影響還要實測，不能只靠這一格代價給出所有模型都會如何的量化結論。

真訓練也會依錯誤目標求梯度，因此先獨立核對標籤，再用乾淨留出題查內容。已有一個局部對照，在相同十題上正標續訓答對7、四筆形狀錯標答對6，錯標支線平均代價卻略低。少量題與單種子不能量化一般損失，但再次說明低loss不能代替正解。

練習只把最後錯候選分數5改成8，先預測錯標loss繼續下降、但真答案的loss變大，再執行並另算label2核對。這次只改模型偏好，沒有更改算術事實，能直接看到「優化提供目標」與「外部任務正確」的差別。

<details>
<summary>補充：實作約定與原始紀錄</summary>

我們已從[7.13的直接SFT屬性基模](https://birdhackor.github.io/tiny-perceptron-vlm/7.13.html)各複製一份，做這個比較。這份起點從零練900次；兩支都用同樣45筆訓練題再更新300次、每批16筆。一支保留正確答案，另一支把其中四筆的`circle`與`square`對調，約佔8.9%。四筆集中在兩個已教過的屬性家族，並不是隨機把所有任務均勻汙染。問題、更新步數與抽題起點不變，兩支實際都計分33,733個訓練目標；留出題仍按獨立正解檢查。

| 繼續練習的答案 | 驗證完整答對 | 最後檢查完整答對 | 最後檢查平均代價 |
| --- | ---: | ---: | ---: |
| 全部保留正解 | 2／5 | 7／10 | 0.37101 |
| 四筆形狀答案寫錯 | 1／5 | 6／10 | 0.34698 |

這次錯標支線少答對一題，最後檢查的平均代價卻略低。這並不矛盾：代價看正確前文下各位置的機率，完整答對看模型沿自己的輸出最後是否生成整份答案，[5.8](https://birdhackor.github.io/tiny-perceptron-vlm/5.8.html)已拆過兩者。兩支在全部留出題都會EOS結束，也不能替內容過關。只有一個種子、兩個被改動的家族和很少的新題，這些數字不能推成固定比例的錯標必然造成多少損失；它們提醒我們保存兩種結果，而不是挑較低loss就說資料無害。[實測報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/sft_ablation.json)列出四筆改動，重做入口在[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
